# ocean: fx

fx ocean grid data

```{dropdown}  **Branding suffix of datasets**
Datasets with the folloing branding suffix are shown in this page: \
[time]-[vertical]-[horizontal]-[domain]
```
* ti-u-hxy-sea
* ti-ol-hxy-sea
    - ti: time independent
    - u: ocean surface
    - ol: ocean levels
    - hxy: horizontal curvlinear grids
    - sea: ocean domain

In [ ]:
## Import libraries
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import ListedColormap, BoundaryNorm
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import cmocean
import sys
import os
import glob
from IPython.display import HTML, display

sys.path.append(os.getcwd())
from utils import load_grid_vertex, read_variables, read_compound_names

In [ ]:
# parameters for the cmorized data
cmorout=''                  # root for cmorized data, e.g., '/scratch/$USER/cmorout'
source_id      = ''         # model name, e.g., 'NorESM3-LM'
experiment_id  = ''         # experiment name, e.g., 'historical', 'ssp585', 'piControl'
variant_label  = ''         # variant label, e.g., 'r1i1p1f1'
grid_label     = ''         # grid label, e.g., 'gn', 'gr', 'g999'
version        = ''         # version, e.g., 'v20260601'

In [ ]:
# data path
data_path = os.path.join(cmorout, source_id, experiment_id, version)

# load grid
grid_file = 'data/grid.nc'
lat, lon, clat, clon = load_grid_vertex(grid_file)
with xr.open_dataset(grid_file) as ds:
    pmask = ds['pmask']
    parea = ds['parea']

parea = parea.rename({'y': 'j', 'x': 'i'})
pmask = pmask.rename({'y': 'j', 'x': 'i'})
# load methods for plotting and set defaults
methods =read_variables('data/methods.txt')
#print(methods.keys())

---
```{dropdown}  **List of datasets:** 
List of compound names of datasets that needs to be cmorized. \
Those datasets which are not cmorized have no link.
```


In [ ]:
# load compound names

coords = ('ti-u-hxy-sea', 'ti-u-hxy-u', 'ti-ol-hxy-sea')
cnames = read_compound_names('data/variables.nml')
# examples of compound names:
# cnames = ['ocean.tos.tavg-u-hxy-sea.mon.glb', 'ocean.ficeberg.tavg-u-hxy-sea.mon.glb']
for cname in cnames:
    realm = cname.split('.')[0]
    var = cname.split('.')[1]
    coord = cname.split('.')[2]
    freq = cname.split('.')[3]
    region = cname.split('.')[4]

    if realm != 'ocean' or coord not in coords:
        continue
    else:
        data_file = f"{var}_{coord}_{freq}_{region}_{grid_label}_{source_id}_{experiment_id}_{variant_label}.nc"
        if not glob.glob(os.path.join(data_path, data_file)):
            print(cname)
            continue
        else:
            display(HTML(f'<a href="#{cname}">{cname}</a>'))


---
```{dropdown}  **Dataset validated**
Each validated dataset has some text information and some plots (2D and/or 1D)
```

In [ ]:
# loop through compound names and plot
for cname in cnames:
    mth_vert = 'mean'
    mth_ts = 'mean'
    mth_cmap = 'mpl.colormaps["viridis"]'
    if cname not in methods.keys():
        print(f"{cname} not found in methods.txt, using default methods for plotting.")
    else:
        if methods[cname] is not None:
            if 'vertical' in methods[cname].keys():
                mth_vert = methods[cname]['vertical']

            if 'timeseries' in methods[cname].keys():
                mth_ts = methods[cname]['timeseries']

            if 'cmap' in methods[cname].keys():
                mth_cmap = methods[cname]['cmap']

    realm = cname.split('.')[0]
    var = cname.split('.')[1]
    coord = cname.split('.')[2]
    freq = cname.split('.')[3]
    region = cname.split('.')[4]

    if realm != 'ocean' or coord not in coords:
        continue

    data_file = f"{var}_{coord}_{freq}_{region}_{grid_label}_{source_id}_{experiment_id}_{variant_label}.nc"

    if not glob.glob(os.path.join(data_path, data_file)):
        continue

    #with xr.open_mfdataset(os.path.join(data_path, data_file)) as ds:
    data_file = glob.glob(os.path.join(data_path, data_file))[0]
    with xr.open_dataset(os.path.join(data_path, data_file)) as ds:
        if var in ds:
            data = ds[var]
        else:
            continue

    # remove the las row of grid info if the data has not the last row
    if data.sizes['j'] == 384:
        parea = parea.isel(j=slice(0, 384))
        pmask = pmask.isel(j=slice(0, 384))

    display(HTML(f'<div id="{cname}"></div>'))
    print(f'\033[1m{cname}\033[0m')
    print(f'long name: {data.long_name} ({data.units})')
#    print(f'timeseries aggregation method for 3D->1D plot: {mth_ts}')
    print(f'original_name: {data.attrs["original_name"]} -> {var}')
    if 'history' in data.attrs:
        print(f'history: {data.attrs["history"]}')
    if 'comment' in data.attrs:
        print(f'comment: {data.attrs["comment"]}')

    if coord == 'ti-ol-hxy-sea':
        if mth_vert == 'mean':
            data2d = data.mean(dim='lev',keep_attrs=True).where(pmask == 1)
        elif mth_vert == 'sum':
            data2d = data.sum(dim='lev',keep_attrs=True).where(pmask == 1)
        else:
            raise ValueError(f'Unsupported vertical aggregation method: {mth_vert}')
        print(f'vertical aggregation method for 3D->2D plot: {mth_vert}')
    else:
        data2d = data 


    #ax1 = fig.add_subplot(1, 2, 1, projection=proj)
    #fig, (ax1, ax2) = plt.subplots( nrows=1, ncols=2, figsize=(12, 5), gridspec_kw={'width_ratios': [2, 1]}, subplot_kw={'projection': proj})

    proj = ccrs.PlateCarree(central_longitude=90.0)
    fig, ax = plt.subplots(1, figsize=(11.7, 4), dpi=96, subplot_kw={"projection": proj})

    # plot 2D map
    #ax1 = fig.add_subplot(gs[0], projection=proj)
    # rescale
    vmin, vmax = data2d.quantile([0.001, 0.999], dim=None).values
    if mth_cmap == 'cmocean.cm.balance':
        vmax = max(abs(vmin),abs(vmax))
        vmin = -vmax
    # Define the discrete levels (boundaries) and the corresponding colors
    levels = np.linspace(vmin, vmax, 21)
    n_levels = 20
    colors = eval(mth_cmap)(np.linspace(0,1,n_levels))
    cmap = ListedColormap(colors)

    # Create a BoundaryNorm to map data to discrete color indices
    norm = BoundaryNorm(levels, ncolors=len(cmap.colors), clip=False)
    
    # Plot the mesh
    if vmin == vmax:
        pm = ax.pcolormesh(lon, lat, data2d, cmap=cmap,
                            transform=ccrs.PlateCarree(), shading='auto', rasterized=True)
    else:
        pm = ax.pcolormesh(lon, lat, data2d, cmap=cmap, norm=norm,
                            transform=ccrs.PlateCarree(), shading='auto', rasterized=True)

    # Add map features
    #ax.stock_img()
    ax.add_feature(cfeature.LAND, facecolor='lightgray')

    gl = ax.gridlines(ylocs=range(-90, 90, 30), draw_labels=True)
    gl.ylocator = mpl.ticker.FixedLocator(range(-90,90,30))

    # Add colorbar
    cb = plt.colorbar(pm, ax=ax, fraction=0.4, shrink=0.8, label='[yr]')
    cb.set_label(label=data.units, size=14)
    cb.ax.tick_params(labelsize=12)
    plt.tight_layout()
        
    #fig, ax = plot_map2d(lon, lat, data2d.mean(dim='time'), proj='PlateCarree', norm=None, cmap=eval(mth_cmap))
    ax.coastlines(resolution='110m')
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    plt.title(data2d.long_name)


    plt.title(data2d.long_name)
    plt.show()

    del data, data2d
    del ax, pm, cb, gl, fig